In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from SynthSpread.spreadviewer_class import SpreadSingle, SpreadViewerData, norm_coeff
#from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Database.DB_reader import Database
from datetime import date, timedelta

from Strategies.LeadLagXGB.backtest_class import BacktestLL
from Strategies.LeadLagXGB.strategy_class import StrategyLL, VolumeClass

from datetime import datetime, time
from Database.TPData import TPData, TPDataDa
from OrderBook.OrderBook import OrderBookSnaps
from SynthSpread.spreadviewer_class import SpreadSingle
from Strategies.Sparse_momentum.ob_attributes import OB_attributes, TR_attributes
import pandas as pd
from Utilities.excel_loaders import conn_out_xload_mac
from Utilities.dfutils import dict_iloc
from Utilities.Storage import get_curr_storage_path
from Utilities.func_utils import load_arguments

In [2]:
dates_out = conn_out_xload_mac()
print("Dates out", dates_out)
STORAGE_ = get_curr_storage_path()
l_path = STORAGE_ + 'Data/orderbooks/base/'

def load_ob(m, t, dt, p_d, bT, eT):
    ob_class = OrderBookSnaps(verbose=True)
    file_path = l_path + t.split('_')[0] + '/'
    file_name = m + '_' + t.split('_')[0] + '_' + p_d.strftime('%y%m%d') + '_' + dt.strftime('%y%m%d') + '.p'
    print('%s Loading OrderBook %d...' % (dt.strftime('%y-%m-%d'), 0))
    time_load = ob_class.import_data(file_path + file_name)
    print('OrderBook %d created in %d sec' % (0, time_load))
    # ob_class.LoB_truncate(thres_vol=1)
    return ob_class.LoB_select(bT, eT, freq=None)
tol=(1e-1)/2

Dates out [datetime.datetime(2024, 1, 1, 0, 0), datetime.datetime(2024, 1, 11, 0, 0), datetime.datetime(2024, 1, 12, 0, 0), datetime.datetime(2024, 1, 15, 0, 0), datetime.datetime(2024, 1, 16, 0, 0), datetime.datetime(2024, 1, 17, 0, 0), datetime.datetime(2024, 1, 18, 0, 0), datetime.datetime(2024, 1, 19, 0, 0), datetime.datetime(2024, 1, 22, 0, 0), datetime.datetime(2024, 1, 23, 0, 0), datetime.datetime(2024, 3, 13, 0, 0), datetime.datetime(2024, 3, 14, 0, 0), datetime.datetime(2024, 3, 15, 0, 0), datetime.datetime(2024, 3, 18, 0, 0), datetime.datetime(2024, 3, 29, 0, 0), datetime.datetime(2024, 4, 1, 0, 0), datetime.datetime(2024, 4, 9, 0, 0), datetime.datetime(2024, 4, 10, 0, 0), datetime.datetime(2024, 5, 1, 0, 0), datetime.datetime(2024, 5, 22, 0, 0), datetime.datetime(2024, 5, 23, 0, 0), datetime.datetime(2024, 5, 24, 0, 0), datetime.datetime(2024, 5, 27, 0, 0), datetime.datetime(2024, 6, 3, 0, 0), datetime.datetime(2024, 10, 25, 0, 0), datetime.datetime(2024, 11, 14, 0, 0), date

In [3]:
def variables_from_instrument(instrument: str):
    result = {
        'mkt': None,
        'tenor': None,
        'tn': None
    }
    for x in ['de', 'fr', 'ttf']:
        if x in instrument:
            result['mkt'] = x
    result['tenor'] = instrument[-2]
    result['tn'] = int(instrument[-1])
    return result

In [4]:
_INSTRUMENT = 'dem2'

In [5]:
ins_dict = variables_from_instrument(_INSTRUMENT)
n_s = 2
mkt_list = [ins_dict['mkt']]
tenor_list = [ins_dict['tenor']]
tn1_list = [ins_dict['tn']]

In [6]:
start_date='2024-07-01'
end_date='2025-06-10'
start_time = time(8, 0, 0, 0)
end_time = time(17, 40, 0, 0)
ins_dict = variables_from_instrument(_INSTRUMENT)
n_s = 2
mkt_list = [ins_dict['mkt']]
tenor_list = [ins_dict['tenor']]
tn1_list = [ins_dict['tn']]
ts_lag = (lambda i: mkt_list[i] + tenor_list[i] + str(tn1_list[i]))(0)

tn2_list = []
prod = 'base'
venue_list = ['eex']
start_date = datetime.strptime(start_date, '%Y-%m-%d').date()
end_date = datetime.strptime(end_date, '%Y-%m-%d').date()

if not tn2_list:
    tn_list = [str(t1) for t1 in tn1_list]
else:
    tn_list = [str(t1) + '_' + str(t2) for (t1, t2) in zip(tn1_list, tn2_list)]

dates = pd.date_range(start_date, end_date, freq='B')

spread_class = SpreadSingle(mkt_list, tenor_list, tn1_list, tn2_list, venue_list)
product_date1 = spread_class.product_dates(dates, n_s, tn_bool=True)
product_date2 = spread_class.product_dates(dates, n_s, tn_bool=False)

In [7]:
def prepare_data_combined(LoB_dicts, obAtt, depth_list, curr_date, inst_ts):
    
    print("Prepare data combined entry")
    timestamp = inst_ts[inst_ts.dt.date == curr_date.date()]

    ob_dict = {k: {} for k in LoB_dicts.keys()}
    for i, LoB in LoB_dicts.items():
        ob_dict[i] = obAtt.prepare_ob_data(LoB, depth_list, aonn=True)    
    ob_df = pd.DataFrame(ob_dict[i])
    # First, create the inst_ts timestamp from df
    # Create a union of all timestamps
    union_ts = pd.Index(timestamp).union(ob_df['timestamp']).drop_duplicates()

    # Set timestamp as index in ob_df for reindexing
    ob_df_indexed = ob_df.set_index('timestamp')

    # Reindex ob_df to union_ts and forward fill
    ob_df_reindexed = ob_df_indexed.reindex(union_ts).ffill().reindex(timestamp)

    # trades attributes
    return ob_df_reindexed

In [8]:
data_class = TPData()
depth_list = [0.0, .3, 0.5, 0.9]
obAtt = OB_attributes(OB_attributes.attr_list())
df_data = pd.DataFrame()
allwd_broker_ids = [1441]

# Selecting all trades since 2024

In [9]:
# 1. Read data from source DB
conn = Database('timescaledb')

query=f"""select distinct datetime, nanotime, tradeid from  public.trades 
          where datetime>='{start_date}' and datetime<='{end_date}' 
          and EXTRACT(HOUR FROM datetime) BETWEEN 8 AND 18
          and instid in ('10641710', '10001075', '10100480', '10012528', '10002806')
          order by datetime asc, nanotime asc""" # where rownum <= 100"""
df=conn.execute(query)


print(f"✅ Loaded {len(df)} rows from source database.")

Connected to the database timescaledb
Disconnected from the database timescaledb
✅ Loaded 4205143 rows from source database.


In [10]:
df['date'] = df['datetime'].dt.date
df = df[~df['date'].isin([x.date() for x in dates_out])]

In [11]:
inst_ts = df['datetime'] + pd.to_timedelta(df['nanotime'].astype(float), unit='ns')
df_ob = pd.DataFrame()

for k, ds in enumerate(dates):
        if ds in dates_out:
            continue
        try:
            LoB_dicts = {}
            tr_dicts = {}
            bT = datetime.combine(ds, start_time)
            eT = datetime.combine(ds, end_time)
            pd1_aux = [None if p is None else p[k] for p in product_date1]
            pd2_aux = [None if p is None else p[k] for p in product_date2]
            for (m, t, n, pd1, pd2) in zip(mkt_list, tenor_list, tn_list,
                                                        pd1_aux, pd2_aux):
                i = m + t + str(n)
                # Order Book attributes
                ob_class = OrderBookSnaps(verbose=True)
                LoB, ts = load_ob(m, t, bT, pd1, bT, eT)
                ob_class.update_data(LoB, ts)

                LoB_dicts[i] = ob_class.LoB_dict
            
            df_current = prepare_data_combined(LoB_dicts, obAtt, depth_list, ds, inst_ts)

            if df_ob.empty:
                df_ob = df_current
            else:
                df_ob = pd.concat([df_ob, df_current])
        except Exception as e:
            print(e)
            pass

24-07-01 Loading OrderBook 0...
OrderBook 0 created in 4 sec
Prepare data combined entry
24-07-02 Loading OrderBook 0...
OrderBook 0 created in 5 sec
Prepare data combined entry
24-07-03 Loading OrderBook 0...
OrderBook 0 created in 4 sec
Prepare data combined entry
24-07-04 Loading OrderBook 0...
OrderBook 0 created in 5 sec
Prepare data combined entry
24-07-05 Loading OrderBook 0...
OrderBook 0 created in 3 sec
Prepare data combined entry
24-07-08 Loading OrderBook 0...
OrderBook 0 created in 4 sec
Prepare data combined entry
24-07-09 Loading OrderBook 0...
OrderBook 0 created in 5 sec
Prepare data combined entry
24-07-10 Loading OrderBook 0...
OrderBook 0 created in 5 sec
Prepare data combined entry
24-07-11 Loading OrderBook 0...
OrderBook 0 created in 5 sec
Prepare data combined entry
24-07-12 Loading OrderBook 0...
OrderBook 0 created in 2 sec
Prepare data combined entry
24-07-15 Loading OrderBook 0...
OrderBook 0 created in 3 sec
Prepare data combined entry
24-07-16 Loading Orde

In [12]:
# Create a mapping dataframe with inst_ts, nanotime, and tradid
trades_mapping = df.reset_index(drop=True)

# Merge the reindexed ob_df with the trades mapping
result = pd.concat([df_ob.reset_index(drop=True), trades_mapping], axis=1)

In [13]:
result['ba_spread'] = result.eval('a_price - b_price')

# Saving into TimescaleDB


In [14]:
result.columns

Index(['ba_spread', 'b_price', 'a_price', 'mid_price', 'b_vol', 'a_vol',
       'mid_priceW_00', 'mid_priceW_30', 'mid_priceW_50', 'mid_priceW_90',
       'ba_volrat_00', 'ba_volrat_30', 'ba_volrat_50', 'ba_volrat_90',
       'mid_priceW_d_00', 'mid_priceW_d_30', 'mid_priceW_d_50',
       'mid_priceW_d_90', 'b_price_sparsity', 'a_price_sparsity', 'datetime',
       'nanotime', 'tradeid', 'date'],
      dtype='object')

In [15]:
columns_dict = ['ba_spread', 'b_price', 'a_price', 'mid_price', 'b_vol',
       'a_vol', 'mid_priceW_00', 'mid_priceW_30', 'mid_priceW_50',
       'mid_priceW_90', 'ba_volrat_00', 'ba_volrat_30', 'ba_volrat_50',
       'ba_volrat_90', 'mid_priceW_d_00', 'mid_priceW_d_30', 'mid_priceW_d_50',
       'mid_priceW_d_90', 'b_price_sparsity', 'a_price_sparsity']

# Insert predictor template

In [16]:
from sqlalchemy import text

# Assuming Database is a custom class or connection handler for TimescaleDB
conn = Database('timescaledb')
i = 100
for pred_name in columns_dict:
    pred_id = i
    pred_name = 'obook' + '_' + pred_name + '_' + _INSTRUMENT
    main_contract = _INSTRUMENT
    description = "Order book prediktor, ffillnuty na casy tradov"
    location = r"EnergyTrading\Python\prefect_data_orchestrator\datamart_inject\obook_predictors.ipynb"
    prod_strategy = None
    author = "MartinScasny"
    additional = "Okamzite hodnoty ffill na timestamp "

    # Define the SQL INSERT statement as a string with placeholders
    stmt = """
    INSERT INTO public.experimental_predictors
    (pred_name, main_contract, description, "location", prod_strategy, author, additional, created_at)
    VALUES (:pred_name, :main_contract, :description, :location, :prod_strategy, :author, :additional, CURRENT_TIMESTAMP)
    """

    # Execute the query with parameters passed as a dictionary
    conn.execute_general_query(stmt, {
        'pred_name': pred_name,
        'main_contract': main_contract,
        'description': description,
        'location': location,
        'prod_strategy': prod_strategy,
        'author': author,
        'additional': additional
    })

    i += 1

print("Data insertion completed successfully.")

Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database timescaledb
Disconnected from the database timescaledb
Connected to the database ti

In [17]:
for pred_name in columns_dict:
    get_pred_id_query = "SELECT pred_id FROM public.experimental_predictors WHERE pred_name = :pred_name"
    pred_id_result = conn.execute_general_query(get_pred_id_query, {'pred_name': 'obook_' + pred_name + '_' + _INSTRUMENT})
    component_pred_id = pred_id_result.iloc[0]['pred_id'] if not pred_id_result.empty else None

    if component_pred_id is None:
        print(f"Error: Could not retrieve pred_id for {pred_name}. Skipping data insertion.")

    print(f"Retrieved pred_id: {component_pred_id} for predictor {pred_name}")
    
    results=[]
    # Step 1: Convert wide df to long format using melt
    df_long = result.melt(
        id_vars=['datetime', 'nanotime', 'tradeid'],  # Keep these columns fixed
        value_vars=[pred_name],
        var_name='pred_name',
        value_name='pred_value'
    )     
    # Step 2: Add pred_name and additional columns if required
    df_long['pred_id'] = component_pred_id

    # Step 3: Sort by datetime for proper forward filling
    df_long = df_long.sort_values(by=['datetime', 'nanotime','pred_id'], ascending=[True, True, True])

    # Optionally drop rows where pred_value is still NaN after forward fill
    #df_long = df_long.dropna(subset=['pred_value'])

    # Step 5: Drop helper colum
    # Optional step: enforce data types explicitly
    df_long['pred_value'] = df_long['pred_value'].astype(float)

    df_long=df_long[['datetime', 'nanotime', 'tradeid', 'pred_id','pred_value']].reset_index(drop=True)


    # 2. Connect to TimescaleDB
    batch_size=100_000
    conn = Database('timescaledb')
    conn._connect()

    # 3. Insert in batches
    total_rows = len(df_long)
    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        batch = df_long.iloc[start:end]

        batch.to_sql('experimental_dataset_entries', conn.engine,schema='public', index=False, if_exists='append',method='multi')
        print(f"✅ Inserted rows {start} to {end} into TimescaleDB.")

        print("🎉 All batches inserted successfully.")

Connected to the database timescaledb
Disconnected from the database timescaledb
Retrieved pred_id: 356 for predictor ba_spread
Connected to the database timescaledb
✅ Inserted rows 0 to 100000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 100000 to 200000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 200000 to 300000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 300000 to 400000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 400000 to 500000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 500000 to 600000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 600000 to 700000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 700000 to 800000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 800000 to 900000 into TimescaleDB.
🎉 All batches inserted successfully.
✅ Inserted rows 900000 to 1000000 into Timescal